# Bài 15: Ảnh giả, giọng giả
**Mở hộp đen AI · Tập 2 · Phòng thí nghiệm Colab**

Ô 1 đến ô 7 là phần lõi, chỉ dùng NumPy và scikit-learn. Ô 8 là phần **mở rộng, kết quả tuỳ ảnh của em**. Ô 3 cần khoảng nửa phút để chạy.
Ebook Bên trong cỗ máy Trí tuệ nhân tạo - Tập 2. Phát hành tại https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Kho ảnh chữ số quen thuộc

In [ ]:
import numpy as np
import warnings
from sklearn.datasets import load_digits
warnings.filterwarnings("ignore")    # ẩn cảnh báo, như Bài 10

X = load_digits().data / 16          # điểm ảnh từ 0 đến 1
print("Số ảnh:", X.shape[0], "| mỗi ảnh:", X.shape[1], "điểm ảnh")

def ve(anh):                           # vẽ ảnh 8 x 8 bằng ký tự
    nen = " .:-=+*#%@"
    for hang in anh.reshape(8, 8):
        print("".join(nen[int(np.clip(v, 0, 1) * 9)] for v in hang))

ve(X[0])

### Ô 2. Trộn ảnh với nhiễu
`s = 0`: ảnh gốc. `s = 1`: toàn nhiễu.

In [ ]:
rng = np.random.default_rng(0)

def tron(anh, s, nhieu):
    return (1 - s) * anh + s * nhieu

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

nhieu = rng.normal(0, 1, 64)
for s in [0, 0.3, 0.6, 0.9, 1]:
    giong = cosine(tron(X[0], s, nhieu), X[0])
    print(f"s = {s}: giống ảnh gốc {giong:5.2f}")

### Ô 3. Dạy máy khử nhiễu
Máy nhìn ảnh đã trộn nhiễu (kèm mức s) và đoán ảnh gốc. Mỗi lần học là một ảnh, một mức nhiễu, một đám nhiễu khác nhau.

In [ ]:
from sklearn.neural_network import MLPRegressor

N = 20_000
chon = rng.integers(0, len(X), N)
s = rng.uniform(0, 1, (N, 1))
nhieu = rng.normal(0, 1, (N, 64))
# đầu vào: 64 điểm ảnh đã trộn nhiễu, thêm 1 cột mức nhiễu
dau_vao = np.hstack([tron(X[chon], s, nhieu), s])

khu_nhieu = MLPRegressor(hidden_layer_sizes=(256, 256),
                         max_iter=60, random_state=0)
khu_nhieu.fit(dau_vao, X[chon])

for muc in [0.3, 0.6, 0.9]:
    vao = np.hstack([tron(X[:200], muc, rng.normal(0, 1, (200, 64))),
                     np.full((200, 1), muc)])
    sai = np.abs(khu_nhieu.predict(vao) - X[:200]).mean()
    print(f"Nhiễu {muc}: sai trung bình mỗi điểm ảnh {sai:.3f}")

### Ô 4. Từ nhiễu thuần tuý, “vẽ” ra chữ số chưa từng có
Bắt đầu từ s = 1 (toàn nhiễu). Mỗi bước: đoán ảnh sạch, rồi trộn lại với ít nhiễu hơn một chút.

In [ ]:
def sinh_anh(so_anh, so_buoc=20, hat_giong=1):
    r = np.random.default_rng(hat_giong)
    x = r.normal(0, 1, (so_anh, 64))                  # bắt đầu: toàn nhiễu
    muc = np.linspace(1, 0, so_buoc + 1)
    for a, b in zip(muc[:-1], muc[1:]):
        vao = np.hstack([x, np.full((so_anh, 1), a)])
        anh_sach = khu_nhieu.predict(vao)
        phan_nhieu = (x - (1 - a) * anh_sach) / a
        x = (1 - b) * anh_sach + b * phan_nhieu   # bớt nhiễu
    return x

anh_moi = sinh_anh(4)
for anh in anh_moi:
    ve(anh)
    print("--------")

## Chặng 2: Đọc
1. Ở ô 2, độ giống ảnh gốc giảm thế nào khi s tăng?
2. Ở ô 3, mức nhiễu nào khó khử nhất? Vì sao?
3. Ở ô 4, máy có “chép” một ảnh có sẵn trong kho không? Làm sao kiểm tra?

## Chặng 3: Sửa
### Ô 5. Dạy một máy khác phát hiện ảnh giả

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

anh_gia = sinh_anh(600, so_buoc=20, hat_giong=2)
anh_that = X[rng.choice(len(X), 600, replace=False)]
D = np.vstack([anh_that, anh_gia])
nhan = np.r_[np.zeros(600), np.ones(600)]                 # 1 = giả
Da, Db, na, nb = train_test_split(D, nhan, test_size=0.3,
                                  random_state=0, stratify=nhan)

phat_hien = MLPClassifier(hidden_layer_sizes=(64,), max_iter=2000,
                          random_state=0)
phat_hien.fit(Da, na)
print(f"Đúng trên đề thi: {phat_hien.score(Db, nb):.1%}")
bat = phat_hien.predict(Db[nb == 1]).mean()
nham = phat_hien.predict(Db[nb == 0]).mean()
print(f"Bắt được ảnh giả (recall): {bat:.1%}")
print(f"Báo nhầm ảnh thật là giả: {nham:.1%}")

### Ô 6. Kẻ gian đổi cách làm giả

In [ ]:
nhanh = sinh_anh(300, so_buoc=3, hat_giong=3)    # chỉ 3 bước
i, j = rng.integers(0, len(X), 300), rng.integers(0, len(X), 300)
ghep = np.maximum(X[i], X[j])                     # chồng 2 ảnh thật
cu = sinh_anh(300, so_buoc=20, hat_giong=5)       # cách cũ, ảnh mới
for ten, Z in [("Giả kiểu đã học (20 bước)", cu), ("Giả kiểu mới (3 bước)", nhanh),
               ("Ảnh ghép hai chữ số", ghep)]:
    print(f"{ten:26}: bắt được {phat_hien.predict(Z).mean():.1%}")


### Ô 7. Ngoài đời, ảnh giả hiếm hơn nhiều
Giả sử cứ 1.000 video thì có 1 video giả. Dùng đúng hai tỉ lệ đo ở ô 5.

In [ ]:
recall = phat_hien.predict(Db[nb == 1]).mean()
bao_nham = phat_hien.predict(Db[nb == 0]).mean()
gia, that = 1, 999
bat_dung = gia * recall
bat_nham = that * bao_nham
print(f"Bị gắn cờ 'giả': {bat_dung + bat_nham:.1f} video")
print(f"Trong đó thật sự giả: {bat_dung:.1f}")
print(f"Precision: {bat_dung / (bat_dung + bat_nham):.1%}")

## Mở rộng: kết quả tuỳ ảnh của em
### Ô 8. Đọc siêu dữ liệu của một bức ảnh
Tải lên một ảnh em tự chụp bằng điện thoại (biểu tượng thư mục bên trái, nút Tải lên), đổi tên tệp trong ô dưới.

In [ ]:
from PIL import Image, ExifTags
anh = Image.open("anh_cua_em.jpg")
exif = anh.getexif()
if not exif:
    print("Ảnh không có siêu dữ liệu.")
for ma, gia_tri in exif.items():
    print(ExifTags.TAGS.get(ma, ma), ":", gia_tri)

## Chặng 4: Tự làm
1. Ở ô 4, đổi `so_buoc` thành 1, 5, 50. Ảnh sinh ra thay đổi thế nào?
2. Ở ô 5, thêm 300 ảnh ghép vào dữ liệu học (nhãn 1) rồi chạy lại ô 5 và ô 6. Máy có bắt được ảnh ghép không? Ảnh 3 bước thì sao?
3. Ở ô 7, đổi thành 1 video giả trên 10 video. Precision thay đổi thế nào?

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*